In [1]:
import re
import sys
import json
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from us import states
from census import Census

import useful
from setup import CENSUS_API_KEY

target_years = [2024]

variables: https://api.census.gov/data/2024/acs/acs5/variables.html

In [8]:
def _tract_econ_data(target_years, loud=False):

    prefixes = {
        "B25119_003": "med_inc", # median renter income "universe": "Occupied housing units"

        "B25003_003": "rent_occ", # "universe": "Occupied housing units"
        "B25069_002": "pay_util", # "universe": "Renter-occupied housing units"
        "B25069_003": "no_pay_util" # "universe": ^  ^  ^

    }
    
    output = useful.tract_states_years_variables(
        useful.target_states, target_years, prefixes, 
        loud=loud)
    


    output['test'] = output['pay_util'] + output['no_pay_util'] - output['rent_occ']
    if set(output['test']) != {0}:
        sys.exit("Data error: rent_off != pay_util + no_pay_util.")
    output.drop(columns=['test'], inplace=True)
    
    output = useful.proportion(output, 'pct_pay_util', 'pay_util', 'rent_occ')


    # first_cols = ["year", "GEOID"]
    # last_cols = [col for col in output.columns if col not in first_cols]
    # output = output[first_cols + last_cols]

    return output


def _cousub_econ_data(target_years, loud=False):

    prefixes = {
        "B25119_003": "med_inc", # median renter income "universe": "Occupied housing units"

        "B25003_003": "rent_occ", # "universe": "Occupied housing units"
        "B25069_002": "pay_util", # "universe": "Renter-occupied housing units"
        "B25069_003": "no_pay_util" # "universe": ^  ^  ^

    }
    
    output = useful.cousub_states_years_variables(
        useful.target_states, target_years, prefixes, 
        loud=loud)
    # output['NAME'] = output["name_str"] + ', ' + output["muni_str"] + ', ' + \
    #         output["county_str"] + ' County, ' + output["state_str"]
    # output["nems"] = output["NAME"].apply(useful.check_nems_membership)
    # output.drop(columns=["NAME"], inplace=True)
    
    output['test'] = output['pay_util'] + output['no_pay_util'] - output['rent_occ']
    if set(output['test']) != {0}:
        sys.exit("Data error: rent_off != pay_util + no_pay_util.")
    output.drop(columns=['test'], inplace=True)
    

    first_cols = ["year", "GEOID"]
    last_cols = [col for col in output.columns if col not in first_cols]
    output = output[first_cols + last_cols]

    return output


In [10]:
cousub_gdf =  gpd.GeoDataFrame(
    pd.concat([
        gpd.read_file('../shapes/tl_2024_09_cousub'),
        gpd.read_file('../shapes/tl_2024_23_cousub'), 
        gpd.read_file('../shapes/tl_2024_25_cousub'),
        gpd.read_file('../shapes/tl_2024_33_cousub'),
        gpd.read_file('../shapes/tl_2024_44_cousub'),
        gpd.read_file('../shapes/tl_2024_50_cousub'),
    ])
).merge(
    _cousub_econ_data(target_years,loud=False), 
    on="GEOID", 
    how="inner"
).reset_index()

In [4]:
round( 100.00*cousub_gdf['pay_util'].sum()/cousub_gdf['rent_occ'].sum(), 2)

np.float64(83.51)

In [ ]:
fig, ax = plt.subplots(figsize=(10,10))
ax.axis('off')

cousub_gdf.loc[
    (cousub_gdf['nems'] == False) &
    (cousub_gdf['med_inc'] > 0)
    ].plot(
      ax=ax, column='med_inc', 
      cmap='Purples', 
      legend=True)

In [22]:
cousub_gdf.to_file("../shapes_out/econ")

/var/folders/tl/gk43p8c13s9_wstnvb4prcf80000gn/T/ipykernel_46784/53323512.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  cousub_gdf.to_file("../shapes_out/econ")
/opt/anaconda3/envs/py310/lib/python3.10/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'no_pay_util' to 'no_pay_uti'
  ogr_write(
/opt/anaconda3/envs/py310/lib/python3.10/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'no_pay_util_m' to 'no_pay_u_1'
  ogr_write(
/opt/anaconda3/envs/py310/lib/python3.10/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'county subdivision' to 'county sub'
  ogr_write(
